# E080 -- Arctic Sea Ice Collapse

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaulVanCode/protoscience-nasa-experiments/blob/main/notebooks/e080_arctic_ice.ipynb)

**Objective:** Download NSIDC Arctic sea ice extent data for September (the annual minimum), fit linear and quadratic trend models, compute the rate of decline per decade, and project when an ice-free Arctic summer may occur.

September Arctic sea ice extent has been declining dramatically since satellite observations began in 1979. This is one of the most visible indicators of climate change, with profound implications for polar ecosystems, global weather patterns, and sea level rise.

## Data Source

- **Institution:** National Snow and Ice Data Center (NSIDC)
- **Dataset:** Sea Ice Index v4.0 — Northern Hemisphere September monthly extent
- **URL:** `https://noaadata.apps.nsidc.org/NOAA/G02135/north/monthly/data/N_09_extent_v4.0.csv`
- **Columns:** year, mo, data_type, region, extent (10^6 km^2), area
- **License:** Public domain (NOAA/NSIDC)

In [ ]:
import urllib.request
import numpy as np
from scipy import stats
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# Download NSIDC September Arctic sea ice extent
url = "https://noaadata.apps.nsidc.org/NOAA/G02135/north/monthly/data/N_09_extent_v4.0.csv"
print("Downloading NSIDC Arctic sea ice data...")
req = urllib.request.Request(url, headers={'User-Agent': 'ProtoScience/1.0'})
response = urllib.request.urlopen(req, timeout=60)
raw = response.read().decode('utf-8')
lines = raw.strip().split('\n')
print(f"Downloaded {len(lines)} lines")
print(f"First lines:")
for line in lines[:5]:
    print(f"  {line}")

In [ ]:
# Parse CSV — find header and extract year + extent
years = []
extents = []
areas = []

header_found = False
for line in lines:
    line = line.strip()
    if not line:
        continue
    # Skip comment/header lines
    parts = [p.strip() for p in line.split(',')]
    try:
        year = int(float(parts[0]))
        if year < 1900 or year > 2100:
            continue
        # extent is typically column 4 (index 4) but let's find it
        extent = None
        for p in parts[2:]:
            try:
                val = float(p)
                if 1.0 < val < 20.0:  # reasonable extent in 10^6 km^2
                    extent = val
                    break
            except ValueError:
                continue
        if extent is not None:
            years.append(year)
            extents.append(extent)
    except (ValueError, IndexError):
        if 'year' in line.lower() or 'extent' in line.lower():
            print(f"Header: {line}")
        continue

years = np.array(years)
extents = np.array(extents)

print(f"\nParsed {len(years)} September extent values")
print(f"Year range: [{years.min()}, {years.max()}]")
print(f"Extent range: [{extents.min():.2f}, {extents.max():.2f}] million km²")
print(f"Record low: {years[np.argmin(extents)]} ({extents.min():.2f} million km²)")

## Trend Analysis

We fit both linear and quadratic models to test whether the decline is accelerating. An ice-free Arctic is conventionally defined as September extent < 1.0 million km^2.

In [ ]:
# Linear fit
slope, intercept, r_value, p_value, std_err = stats.linregress(years, extents)
decline_per_decade = slope * 10

# Quadratic fit
quad_coeffs = np.polyfit(years, extents, 2)
quad_func = np.poly1d(quad_coeffs)

# Residual comparison
resid_lin = extents - (slope * years + intercept)
resid_quad = extents - quad_func(years)
rmse_lin = np.sqrt(np.mean(resid_lin**2))
rmse_quad = np.sqrt(np.mean(resid_quad**2))

print("=== Linear Trend ===")
print(f"  Slope: {slope:.4f} million km²/year")
print(f"  Decline per decade: {decline_per_decade:.2f} million km²")
print(f"  R² = {r_value**2:.4f}")
print(f"  p-value = {p_value:.2e}")
print(f"  RMSE = {rmse_lin:.3f} million km²")

print(f"\n=== Quadratic Trend ===")
print(f"  Coefficients: a={quad_coeffs[0]:.6f}, b={quad_coeffs[1]:.4f}, c={quad_coeffs[2]:.1f}")
print(f"  RMSE = {rmse_quad:.3f} million km²")
if quad_coeffs[0] < 0:
    print(f"  Acceleration: YES (negative curvature = decline speeding up)")

# Ice-free projection
# Linear: year where extent = 1.0
year_icefree_lin = (1.0 - intercept) / slope
print(f"\n=== Ice-Free Projection (extent < 1.0 million km²) ===")
print(f"  Linear model: ~{year_icefree_lin:.0f}")

# Quadratic: solve a*y^2 + b*y + c = 1.0
roots = np.roots([quad_coeffs[0], quad_coeffs[1], quad_coeffs[2] - 1.0])
future_roots = [r for r in roots if r > years.max() and np.isreal(r)]
if future_roots:
    year_icefree_quad = min(r.real for r in future_roots)
    print(f"  Quadratic model: ~{year_icefree_quad:.0f}")
else:
    year_icefree_quad = None
    print(f"  Quadratic model: no future crossing found")

In [ ]:
# Decade averages
decades = np.arange(1980, 2040, 10)
decade_means = []
decade_labels = []
for d in decades:
    m = (years >= d) & (years < d + 10)
    if m.sum() > 0:
        decade_means.append(extents[m].mean())
        decade_labels.append(f"{d}s")

print("\n=== Decade Averages ===")
for label, mean in zip(decade_labels, decade_means):
    print(f"  {label}: {mean:.2f} million km²")

# === 4-SUBPLOT FIGURE ===
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle("E080: Arctic Sea Ice Collapse — September Minimum Extent",
             fontsize=15, fontweight='bold')

# (a) Time series with fits
ax = axes[0, 0]
ax.scatter(years, extents, s=40, color='steelblue', zorder=5, edgecolor='black', linewidth=0.5)
y_proj = np.arange(years.min(), max(2060, year_icefree_lin + 5))
ax.plot(y_proj, slope * y_proj + intercept, 'r--', linewidth=2,
        label=f'Linear ({decline_per_decade:.2f}/decade)')
ax.plot(y_proj, quad_func(y_proj), 'orange', linewidth=2, linestyle='-.',
        label='Quadratic')
ax.axhline(1.0, color='darkred', linestyle=':', linewidth=2, label='Ice-free threshold')
ax.fill_between(y_proj, 0, 1.0, alpha=0.1, color='red')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('September Extent [10$^6$ km²]', fontsize=11)
ax.set_title('(a) September Arctic Sea Ice Extent', fontsize=12)
ax.legend(fontsize=9)
ax.set_ylim(0, extents.max() * 1.1)

# (b) Rate of change (5-year rolling slope)
ax = axes[0, 1]
window = 5
rolling_slopes = []
rolling_years = []
for i in range(len(years) - window + 1):
    s, _, _, _, _ = stats.linregress(years[i:i+window], extents[i:i+window])
    rolling_slopes.append(s * 10)  # per decade
    rolling_years.append(years[i:i+window].mean())
ax.plot(rolling_years, rolling_slopes, 'steelblue', linewidth=2)
ax.axhline(decline_per_decade, color='red', linestyle='--', linewidth=1.5,
           label=f'Overall: {decline_per_decade:.2f}/decade')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Rate of Change [10$^6$ km²/decade]', fontsize=11)
ax.set_title(f'(b) {window}-Year Rolling Decline Rate', fontsize=12)
ax.legend(fontsize=10)

# (c) Decade averages
ax = axes[1, 0]
colors = plt.cm.RdYlBu(np.linspace(0.8, 0.1, len(decade_means)))
ax.bar(decade_labels, decade_means, color=colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Mean September Extent [10$^6$ km²]', fontsize=11)
ax.set_title('(c) Decade Averages', fontsize=12)
for i, (label, mean) in enumerate(zip(decade_labels, decade_means)):
    ax.text(i, mean + 0.1, f'{mean:.1f}', ha='center', fontsize=10, fontweight='bold')

# (d) Anomaly from 1979-2000 mean
ax = axes[1, 1]
ref_mask = (years >= 1979) & (years <= 2000)
ref_mean = extents[ref_mask].mean() if ref_mask.any() else extents.mean()
anomaly = extents - ref_mean
colors_anom = ['steelblue' if a >= 0 else 'crimson' for a in anomaly]
ax.bar(years, anomaly, color=colors_anom, edgecolor='black', linewidth=0.3)
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Anomaly from 1979-2000 mean [10$^6$ km²]', fontsize=11)
ax.set_title(f'(d) September Extent Anomaly (ref = {ref_mean:.2f})', fontsize=12)

plt.tight_layout()
plt.savefig('e080_arctic_ice.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nFigure saved: e080_arctic_ice.png")

## Key Results Summary

| Analysis | Result |
|----------|--------|
| Record years | Satellite era from 1979 |
| Decline rate | Per decade (linear) |
| Acceleration | Quadratic curvature (if negative) |
| Ice-free projection | Linear and quadratic estimates |
| Total loss since 1979 | Difference from first to last decade average |

**Physical interpretation:** Arctic sea ice loss is driven by rising global temperatures through the ice-albedo feedback: as ice melts, darker ocean absorbs more sunlight, causing further warming and more melting. The September minimum has declined from ~7 million km^2 in the 1980s to ~4 million km^2 in recent years — a loss of ~40%. Current projections suggest ice-free Arctic summers (< 1 million km^2) could occur within decades. This represents one of the most dramatic ongoing changes in the Earth system.